In [30]:
# Benjamin_Graves_Jewish_Names.py
# 4/27/2025

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.by import By
import json
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import sys
import random
from zenrows import ZenRowsClient
import requests
import math
import numpy as np
import urllib3
import inspect
# from IPython.display import display


# def maximize_screen():
#     from IPython.display import display, HTML
#     display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def left(s, amount):
    return s[:amount]
def right(s, amount):
    return s[-amount:]
def mid(s, offset, amount):
    return s[offset:offset+amount]

def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def convert_date_z(date): 
    date_t= pd.to_datetime(date, utc=True)
    date_out=date_t.dt.strftime("%m/%d/%Y")
    return date_out

def convert_datetime_t(event_date_t):
    no_time='2000-01-01T00:00:00'
    no_time_dt=dt.strptime(no_time, "%Y-%m-%dT%H:%M:%S") 
    event_date_t = event_date_t.fillna(no_time)
    event_date_dt=pd.to_datetime(event_date_t, format="%Y-%m-%dT%H:%M:%S")
    event_date_index=event_date_dt.dt.strftime("%Y%m%d")
    event_date =event_date_dt.dt.strftime("%m/%d/%Y")
    event_date_rnd_dt = event_date_dt + timedelta(minutes=30)
    event_time_index=event_date_rnd_dt.dt.strftime('%H')  
    event_time = event_date_dt.dt.strftime('%H:%M')
    return     event_date,event_date_index,event_time,event_time_index 



def random_sleep(call_count): 
    global sleep_time0,listings_df
    if call_count%100==0:
        sleep_time =sleep_time0
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds • Sleeping till: {sleep_end_date}")
        # listings_df.to_csv(fileOut,mode="w", index=False,header=True)
        # listings_df.to_csv(fileOut0,mode="w", index=False,header=True) 
    else:
        sleep_time = random.uniform(2, 4)
    return sleep_time 

def get_total_cemeteries(url):
    global my_header
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    total_cemeteries=capture_json['data']['cemeteries']['total']
    return total_cemeteries

def get_grave_count(url):
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    grave_count=capture_json['total']
    return capture_json,grave_count
    
def extract_grave_data(capture_json):
    global field_list_set,field_list
    collection_df=pd.json_normalize(capture_json['collection'])
    collection_columns_set=set(list(collection_df))
    missing_columns=field_list_set-collection_columns_set
    collection_df[list(missing_columns)]=np.nan
    grave_frame_df=collection_df[field_list]
    return grave_frame_df

def clean_dataframe(graves_df):
    global field_list
    for column in field_list:
        graves_df[column]= graves_df[column].astype(str).replace(r"\,", ' * ', regex=True).str.strip()
        graves_df[column]= graves_df[column].astype(str).replace(r"\'", '', regex=True).str.strip()
    delete_list=[r'\"',r"\'",r'\?',r'\[',r'\]']
    for item in delete_list:
        for column in field_list:
            graves_df[column]= graves_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return graves_df

#*************** Append  *******************************
append=0
# append_flag=input('To Append Press 1')
# if append_flag=='1':append=1
#*************** Append  *******************************

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Extract Benjamin Graves Jewish Extract '
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"


fileInTest="Names_Test"
fileIn="Benjamin_Cemeteries_Jewish"
fileOutput="Benjamin_Graves_Jewish_Names_Test"

# if append==1:fileIn='Benjamin_Cemeteries_Jewish_Test_X'

fileNamesIn=DirMain+"Analysis\\"+fileInTest+".csv"
fileCemeteriesIn=DirMain+"OutputData\\Cemeteries\\"+fileIn+".csv"
fileOut=DirMain+"OutputData\\Graves\\"+fileOutput+"_"+timestamp+".csv"

fileOut0=DirMain+"OutputData\\Graves\\"+fileOutput+".csv"
fileOutInt=DirMain+"OutputData\\Graves\\"+fileOutput+"_Interim.csv"

cemetery_info_df=pd.read_csv(fileCemeteriesIn)
cemetery_id_list=cemetery_info_df["cemetery_id"]
cemetery_list=cemetery_info_df["cemetery"]



names_df=pd.read_csv(fileNamesIn)
name_list=names_df["Name"]
war_list=names_df["War"]

my_header = {
  'priority': 'u=1, i',
  'sec-ch-ua': '"Google Chrome";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
  'sec-ch-ua-platform': '"Windows"',
  'sec-fetch-mode': 'cors',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
  'Cookie': '__cf_bm=O55WB_mmMt72i8KoQuzWliizNTb6J7zg2RVHXqbn.vc-1745620049-1.0.1.1-IDSlPNW4kRjBMAr5n8M2obOoGp9GAod_BHbB997Ze2F0YUm2DvAFh5TaijsQhPWR0iyIyFoaSy.p1YDke07ZttHHC83SJpQCEBFmaeImKIc; _cfuvid=zhvP.IczaAr9sd4YVjWjTqBayGxsei_Ntl0ANgSCCBg-1745620049033-0.0.1.1-604800000; ipLoc=us; preferredLanguage=en'
}

field_list=['lastName','firstName','fullName','birthYear','deathYear','birthMonth','deathMonth','birthDate','deathDate',\
    'cemeteryName','cemeteryId','cityName','stateName','maidenName','Parents','Children', 'Siblings','Spouses']
field_list_set=set(field_list)
if append==1:print('**** Appending ****')    
if append!=1:graves_df=pd.DataFrame([])  

sleep_time0=1*60

date_df=pd.DataFrame([])

get_counter=0
name_count=0
for name in name_list:
    war_name=war_list.iloc[name_count]

    url0 = "https://www.findagrave.com/memorial/search?ajax=true&skip="       
    url1 ="&limit=100&firstname=&middlename=&lastname="
    url2 ="&birthyear=&birthyearfilter=&deathyear=1980&deathyearfilter=before&locationId=&location=&memorialid=&datefilter=&orderby=b&photofilter=&gpsfilter=&famous=&cenotaph=&monument=&isVeteran=&flowers=&sponsored=&noCemetery=&includeNickName=&includeMaidenName=&includeTitles=&hasPlot=&plot=&bio=&partialLastName=&exactName=true&linkedToName=&fuzzyNames=&mcid="
    frame_count=0
    name_count+=1

    grave_count=100000
    grave_frame_count=0
    start_birth_year=1900
    while grave_frame_count<grave_count:
        frame_count+=1
        url=url0+str(grave_frame_count)+url1+name+url2
        capture_json,grave_count=get_grave_count(url)
        
       #   ******************* graves>10,000  ***********************
        if grave_count>9999:
            birth_year=start_birth_year
            interval_baseline=10
            url2_a ="&birthyear="
            url2_b="&birthyearfilter="
            url2_c="&deathyear=&deathyearfilter=&locationId=&location=&memorialid=&datefilter=&orderby=b&photofilter=&gpsfilter=&famous=&cenotaph=&monument=&isVeteran=&flowers=&sponsored=&noCemetery=&includeNickName=&includeMaidenName=&includeTitles=&hasPlot=&plot=&bio=&partialLastName=&exactName=true&linkedToName=&fuzzyNames=&mcid="           
            birthyearfilter='before'
            grave_frame_count=0
            before_high_flag=0
            #   ******************* Starting Before Start Year: graves>10,000   ***********************
            while grave_count>9999:
                birth_year=int(round(birth_year, 0))
                url=url0+str(grave_frame_count)+url1+name+url2_a+str(birth_year)+url2_b+birthyearfilter
                capture_json,grave_count=get_grave_count(url)
                print('Before: grave_count,next birth_year,birthyearfilter',grave_count,birth_year,birthyearfilter)  
                
                birth_year-=10
                if birth_year<=1860:before_high_flag=1;break
                get_counter+=1
                sleep_time=random_sleep(get_counter)       
                time.sleep(sleep_time)        
            birth_year+=10    
            print('Before year processing:- Birth Year:',birth_year)
            #   ******************* "Before" year processing: By Frames of 100   ***********************
            while grave_frame_count<grave_count and before_high_flag==0:
                frame_count+=1
                birth_year=int(round(birth_year, 0))
                url=url0+str(grave_frame_count)+url1+name+url2_a+str(birth_year)+url2_b+birthyearfilter
                capture_json,grave_count=get_grave_count(url)
                grave_frame_df=extract_grave_data(capture_json)
                graves_df = graves_df._append(grave_frame_df, ignore_index = True) 
                grave_frame_count+=len(grave_frame_df)
                graves_df=clean_dataframe(graves_df)
                graves_df.to_csv(fileOutInt,mode="w", index=False,header=True)
                print(f"    --- {process_name:<20}• Name: {name} • War: {war_name} • grave count: {grave_frame_count}/{grave_count} ")
                get_counter+=1
                sleep_time=random_sleep(get_counter)       
                time.sleep(sleep_time)  
            
            interval=interval_baseline
            birth_year+=interval
            birthyearfilter=str(interval)
            #   ******************* Process all subsequent years after the "Before" year  ***********************
            while birth_year<=1950:
                interval=interval_baseline
                birth_year=int(round(birth_year, 0))
                url=url0+str(grave_frame_count)+url1+name+url2_a+str(birth_year)+url2_b+birthyearfilter
                capture_json,grave_count=get_grave_count(url)
                # print('Subsequent year: grave_count,next birth_year,birthyearfilter,next interval',grave_count,birth_year,birthyearfilter,interval) 
                grave_frame_count=0
                get_counter+=1
                sleep_time=random_sleep(get_counter)       
                time.sleep(sleep_time)                  
                #   ******************* Subsequent year interval: graves>10,000  ***********************
                while grave_count>9999:
                    birth_year=int(round(birth_year, 0))
                    url=url0+str(grave_frame_count)+url1+name+url2_a+str(birth_year)+url2_b+birthyearfilter
                    capture_json,grave_count=get_grave_count(url)
            
                    # print('Loop Interval: grave_count,next birth_year,birthyearfilter,next interval',grave_count,birth_year,birthyearfilter,interval)        
                    interval=round(interval/2, 0)
                    if interval==1:print('%%%%%% Interval = 1  %%%%%%%%%%');break
                    get_counter+=1
                    sleep_time=random_sleep(get_counter)       
                    time.sleep(sleep_time)         
                print('Interval processing:- Birth Year:',birth_year)
                # print('Loop Interval: grave_count,next birth_year,birthyearfilter,next interval',grave_count,birth_year,birthyearfilter,interval) 
                # input('Start: Subsequent year processing: By Frames of 100')
                #   ******************* "Subsequent" year processing: By Frames of 100   ***********************
                while grave_frame_count<grave_count and grave_frame_count<10000:
                    frame_count+=1
                    birth_year=int(round(birth_year, 0))
                    url=url0+str(grave_frame_count)+url1+name+url2_a+str(birth_year)+url2_b+birthyearfilter
                    capture_json,grave_count=get_grave_count(url)
                    grave_frame_df=extract_grave_data(capture_json)
                    graves_df = graves_df._append(grave_frame_df, ignore_index = True) 
                    grave_frame_count+=len(grave_frame_df)
                    graves_df=clean_dataframe(graves_df)
                    graves_df.to_csv(fileOutInt,mode="w", index=False,header=True)
                    
                    # print('Frame Processing Loop: grave_count,next birth_year,birthyearfilter,next interval',grave_count,birth_year,birthyearfilter,interval) 
                    print(f"    --- {process_name:<20}• Name: {name} • War: {war_name} • grave count: {grave_frame_count}/{grave_count} ")
                    # input('Next Frame: Subsequent year processing: By Frames of 100')
                    get_counter+=1
                    sleep_time=random_sleep(get_counter)       
                    time.sleep(sleep_time)        
                birth_year+=2*interval
                # print('Loop Year Interval: grave_count,next birth_year,birthyearfilter,next interval',grave_count,birth_year,birthyearfilter,interval) 
        grave_frame_df=extract_grave_data(capture_json)
        graves_df = graves_df._append(grave_frame_df, ignore_index = True) 
        grave_frame_count+=len(grave_frame_df)
        
        graves_df=clean_dataframe(graves_df)
        graves_df.to_csv(fileOutInt,mode="w", index=False,header=True)
        print(f"    --- {process_name:<20}• Name: {name} • War: {war_name} • grave count: {grave_frame_count}/{grave_count} ")
        get_counter+=1
        sleep_time=random_sleep(get_counter)       
        time.sleep(sleep_time) 
    try:    
        last_death_year=int(graves_df['deathYear'][pd.to_numeric(graves_df['deathYear'], errors='coerce')!=9999].max())   
        first_death_year=int(graves_df['deathYear'][pd.to_numeric(graves_df['deathYear'], errors='coerce')!=9999].min())  
    except:
        first_death_year=np.nan
        last_death_year=np.nan
        
    # graves_df.to_csv(fileOutInt,mode="w", index=False,header=True)
    end_time=datetime.datetime.now()
    run_processing_time=end_time-event_time
    event_time=end_time
    run_processing_time_seconds=int(run_processing_time.total_seconds())
    processing_time=end_time-start_time
    processing_time_minutes=int(processing_time.total_seconds())/60  
    
    print(f"*** {process_name:<20}• Name: {name} • War: {war_name} • grave count: {grave_frame_count}  • Death Years Range: {first_death_year}-{last_death_year}\n")

graves_df=clean_dataframe(graves_df)

# for column in field_list:
#         graves_df[column]= graves_df[column].astype(str).replace(r"\*", ' * ', regex=True).str.strip()
#         graves_df[column]= graves_df[column].astype(str).replace(r"\,", ' * ', regex=True).str.strip()
#         graves_df[column]= graves_df[column].astype(str).replace(r"\'", '', regex=True).str.strip()
# delete_list=[r'\"',r"\'",r'\?',r'\[',r'\]']
# for item in delete_list:
#     for column in field_list:
#         graves_df[column]= graves_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
        
end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** {process_name}   Processing Time: {processing_time_seconds} Sec")   

graves_df.to_csv(fileOut,mode="w", index=False,header=True)
graves_df.to_csv(fileOut0,mode="w", index=False,header=True) 
print("\n")
print('Uploaded To: ',fileOut0)


**Starting ***  Extract Benjamin Graves Jewish Extract   *** , Time: 2025-05-06 17:24:19.163452  **

Before: grave_count,next birth_year,birthyearfilter 12275 1900 before
Before: grave_count,next birth_year,birthyearfilter 9494 1890 before
Before year processing:- Birth Year: 1890
    --- Extract Benjamin Graves Jewish Extract • Name: Moser • War: WW I • grave count: 100/9494 
    --- Extract Benjamin Graves Jewish Extract • Name: Moser • War: WW I • grave count: 200/9494 
    --- Extract Benjamin Graves Jewish Extract • Name: Moser • War: WW I • grave count: 300/9494 
    --- Extract Benjamin Graves Jewish Extract • Name: Moser • War: WW I • grave count: 400/9494 
    --- Extract Benjamin Graves Jewish Extract • Name: Moser • War: WW I • grave count: 500/9494 
    --- Extract Benjamin Graves Jewish Extract • Name: Moser • War: WW I • grave count: 600/9494 
    --- Extract Benjamin Graves Jewish Extract • Name: Moser • War: WW I • grave count: 700/9494 
    --- Extract Benjamin Graves 

KeyboardInterrupt: 

In [31]:
name_list

0         Moser
1       Bardman
2          Burd
3      Afremoff
4     Pachtmann
5     Charifson
6        Tamkin
7          Hanf
8        Akabas
9      Sugarman
10      Meltzer
11      Belmont
12      Feldman
13       Papurt
14       Seixas
15      Cordova
16         Fink
Name: Name, dtype: object

In [27]:
def clean_dataframe(graves_df):
    global field_list
    for column in field_list:
        graves_df[column]= graves_df[column].astype(str).replace(r"\*", ' * ', regex=True).str.strip()
        graves_df[column]= graves_df[column].astype(str).replace(r"\,", ' * ', regex=True).str.strip()
        graves_df[column]= graves_df[column].astype(str).replace(r"\'", '', regex=True).str.strip()
    delete_list=[r'\"',r"\'",r'\?',r'\[',r'\]']
    for item in delete_list:
        for column in field_list:
            graves_df[column]= graves_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return graves_df

while birth_year<=1950:
    interval=interval_baseline
    birth_year=int(round(birth_year, 0))
    url=url0+str(grave_frame_count)+url1+name+url2_a+str(birth_year)+url2_b+birthyearfilter
    capture_json,grave_count=get_grave_count(url)
    # print('Subsequent year: grave_count,next birth_year,birthyearfilter,next interval',grave_count,birth_year,birthyearfilter,interval) 
    grave_frame_count=0
    get_counter+=1
    sleep_time=random_sleep(get_counter)       
    time.sleep(sleep_time)                  
    #   ******************* Subsequent year interval: graves>10,000  ***********************
    while grave_count>9999:
        birth_year=int(round(birth_year, 0))
        url=url0+str(grave_frame_count)+url1+name+url2_a+str(birth_year)+url2_b+birthyearfilter
        capture_json,grave_count=get_grave_count(url)

        # print('Loop Interval: grave_count,next birth_year,birthyearfilter,next interval',grave_count,birth_year,birthyearfilter,interval)        
        interval=round(interval/2, 0)
        if interval==1:print('%%%%%% Interval = 1  %%%%%%%%%%');break
        get_counter+=1
        sleep_time=random_sleep(get_counter)       
        time.sleep(sleep_time)         
    print('Interval processing:- Birth Year:',birth_year)
    # print('Loop Interval: grave_count,next birth_year,birthyearfilter,next interval',grave_count,birth_year,birthyearfilter,interval) 
    # input('Start: Subsequent year processing: By Frames of 100')
    #   ******************* "Subsequent" year processing: By Frames of 100   ***********************
    while grave_frame_count<grave_count and grave_frame_count<10000:
        frame_count+=1
        birth_year=int(round(birth_year, 0))
        url=url0+str(grave_frame_count)+url1+name+url2_a+str(birth_year)+url2_b+birthyearfilter
        capture_json,grave_count=get_grave_count(url)
        grave_frame_df=extract_grave_data(capture_json)
        graves_df = graves_df._append(grave_frame_df, ignore_index = True) 
        grave_frame_count+=len(grave_frame_df)
        graves_df=clean_dataframe(graves_df)
        graves_df.to_csv(fileOutInt,mode="w", index=False,header=True)
        
        # print('Frame Processing Loop: grave_count,next birth_year,birthyearfilter,next interval',grave_count,birth_year,birthyearfilter,interval) 
        print(f"    --- {process_name:<20}• Name: {name} • War: {war_name} • grave count: {grave_frame_count}/{grave_count} ")
        # input('Next Frame: Subsequent year processing: By Frames of 100')
        get_counter+=1
        sleep_time=random_sleep(get_counter)       
        time.sleep(sleep_time)        
    birth_year+=2*interval
    # print('Loop Year Interval: grave_count,next birth_year,birthyearfilter,next interval',grave_count,birth_year,birthyearfilter,interval) 



%%%%%% Interval = 1  %%%%%%%%%%
Interval processing:- Birth Year: 1902
    --- Extract Benjamin Graves Jewish Extract • Name: Goodman • War: WW II • grave count: 100/13202 
    --- Extract Benjamin Graves Jewish Extract • Name: Goodman • War: WW II • grave count: 200/13202 
    --- Extract Benjamin Graves Jewish Extract • Name: Goodman • War: WW II • grave count: 300/13202 
    --- Extract Benjamin Graves Jewish Extract • Name: Goodman • War: WW II • grave count: 400/13202 
    --- Extract Benjamin Graves Jewish Extract • Name: Goodman • War: WW II • grave count: 500/13202 
    --- Extract Benjamin Graves Jewish Extract • Name: Goodman • War: WW II • grave count: 600/13202 
    --- Extract Benjamin Graves Jewish Extract • Name: Goodman • War: WW II • grave count: 700/13202 
    --- Extract Benjamin Graves Jewish Extract • Name: Goodman • War: WW II • grave count: 800/13202 
    --- Extract Benjamin Graves Jewish Extract • Name: Goodman • War: WW II • grave count: 900/13202 
    --- Ext

KeyboardInterrupt: 

In [ ]:
len(grave_frame_df)

In [ ]:
start_birth_year